# Olist Data Ecosystem: Do ETL à Análise de Sentimentos: Capítulo 1 - Engenharia de Dados

## 1. Fazendo o Merge das tabelas

In [2]:
# Importando o Pandas
import pandas as pd
print("Iniciando a extração de dados...")
caminho = 'dados/'
# lendo os arquivos CSV
df_pedidos = pd.read_csv(caminho + 'olist_orders_dataset.csv')
df_itens = pd.read_csv(caminho + 'olist_order_items_dataset.csv')
df_avaliacoes = pd.read_csv(caminho + 'olist_order_reviews_dataset.csv')
df_pagamentos = pd.read_csv(caminho + 'olist_order_payments_dataset.csv')
df_clientes = pd.read_csv(caminho + 'olist_customers_dataset.csv')
df_produtos = pd.read_csv(caminho + 'olist_products_dataset.csv')
print("Dados extraídos com sucesso! Iniciando o Merge.....")
# Merge dos DataFrames
df_merge = pd.merge(df_pedidos, df_itens, on='order_id', how='inner') #só conta os pedidos que possuem itens
df_merge = pd.merge(df_merge, df_avaliacoes, on='order_id', how='left') # nem todos os pedidos possuem avaliações, por isso o left
df_merge = pd.merge(df_merge, df_pagamentos, on='order_id', how='left') # nem todos os pedidos possuem pagamentos, por isso o left
df_merge = pd.merge(df_merge, df_clientes, on='customer_id', how='inner') # só conta os clientes que possuem pedidos
df_merge = pd.merge(df_merge, df_produtos, on='product_id', how='left')  # nem todos os itens possuem informações de produtos, por isso o left
print("Merge realizado com sucesso!")
print("Nossa tabela final possui {} linhas e {} colunas.".format(df_merge.shape[0], df_merge.shape[1]))
# Exibindo as primeiras linhas do DataFrame resultante
print(df_merge.head())


Iniciando a extração de dados...
Dados extraídos com sucesso! Iniciando o Merge.....
Merge realizado com sucesso!
Nossa tabela final possui 118310 linhas e 36 colunas.
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
2  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
3  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
4  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
2    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
3    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
4    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   

  order_

## 2. Fazendo a limpeza de dados

In [3]:
# Contando quantos valores nulos existem em cada coluna
print("\nContagem de valores nulos por coluna:")
valores_nulos = df_merge.isnull().sum()
# Mostrando apenas as colunas que possuem valores nulos
colunas_valores_nulos = valores_nulos[valores_nulos > 0].sort_values(ascending=False)
print("Colunas com valores nulos:")
print("-" * 40)
print(colunas_valores_nulos)
print("-" * 40)
# Calculando a porcentagem de avaliações nulas
total_linhas = len(df_merge)
textos_vazios = df_merge['review_comment_message'].isnull().sum()
porcentagem_vazios = (textos_vazios / total_linhas) * 100
print("{}% dos pedidos não possuem avaliações.".format(porcentagem_vazios))


Contagem de valores nulos por coluna:
Colunas com valores nulos:
----------------------------------------
review_comment_title             104418
review_comment_message            68628
order_delivered_customer_date      2588
product_category_name              1709
product_name_lenght                1709
product_photos_qty                 1709
product_description_lenght         1709
order_delivered_carrier_date       1254
review_id                           978
review_creation_date                978
review_score                        978
review_answer_timestamp             978
product_weight_g                     20
product_height_cm                    20
product_length_cm                    20
product_width_cm                     20
order_approved_at                    15
payment_value                         3
payment_installments                  3
payment_type                          3
payment_sequential                    3
dtype: int64
----------------------------------------

In [4]:
print("Iniciando a limpeza dos dados...")
# Preenchendo os textos vazios com "Sem comentário" e "Sem título"
df_merge['review_comment_message'] = df_merge['review_comment_message'].fillna("Sem comentário")
df_merge['review_comment_title'] = df_merge['review_comment_title'].fillna("Sem título")
# Preenchendo a categoria de produto vazia com "Outros"
df_merge['product_category_name'] = df_merge['product_category_name'].fillna("Outros")
# Removendo as linhas onde a encomenda nunca chegou
df_limpo = df_merge.dropna(subset=['order_delivered_customer_date'])
# Removendo os vazios de pagamento e aprovação
df_limpo = df_limpo.dropna(subset=['order_approved_at', 'payment_value'])
print("Limpeza dos dados concluída! A tabela foi de {} linhas e {} colunas para {} linhas e {} colunas.".format(df_merge.shape[0], df_merge.shape[1], df_limpo.shape[0], df_limpo.shape[1]))
print("Conferindo se ainda existem valores nulos...")
print(df_limpo[['review_comment_message', 'review_comment_title', 'product_category_name', 'order_delivered_customer_date', 'order_approved_at', 'payment_value']].isnull().sum())


Iniciando a limpeza dos dados...
Limpeza dos dados concluída! A tabela foi de 118310 linhas e 36 colunas para 115704 linhas e 36 colunas.
Conferindo se ainda existem valores nulos...
review_comment_message           0
review_comment_title             0
product_category_name            0
order_delivered_customer_date    0
order_approved_at                0
payment_value                    0
dtype: int64
